In [ ]:
from dotenv import load_dotenv
import os
from sqlalchemy import create_engine, text

load_dotenv(dotenv_path=".env")

DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DEFAULT_DB = os.getenv("DEFAULT_DB")
NEW_DB = os.getenv("NEW_DB")
DB_NAME = os.getenv("DB_NAME")


default_db_url = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/postgres"
engine = create_engine(default_db_url)



with engine.connect() as conn:
    conn.execution_options(isolation_level="AUTOCOMMIT")
    
    result = conn.execute(text(f"SELECT 1 FROM pg_database WHERE datname = '{DB_NAME}'"))
    
    if not result.scalar():
        conn.execute(text(f"CREATE DATABASE {DB_NAME}"))
        print(f"Database '{DB_NAME}' created successfully.")
    else:
        print(f"Database '{DB_NAME}' already exists.")

In [ ]:
import os
import urllib.parse
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.exc import SQLAlchemyError

load_dotenv(financecore_db.env)

DB_USER = os.getenv("DB_USER")
DB_PASS = os.getenv("DB_PASS")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

encoded_pass = urllib.parse.quote_plus(DB_PASS)
DATABASE_URL = f"postgresql+psycopg://{DB_USER}:{encoded_pass}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(DATABASE_URL)

def run_health_check():
    print("🔍 System integrity check ...")
    try:
        with engine.connect() as conn:
            res = conn.execute(text("SELECT current_user, current_database(), version();"))
            user, db, ver = res.fetchone()
            print(f"✅ Connection: Stable")
            print(f"👤 User     : {user}")
            print(f"🗄️ Database : {db}")

            conn.execute(text("CREATE TEMP TABLE connection_test (id serial PRIMARY KEY, val text);"))
            conn.execute(text("INSERT INTO connection_test (val) VALUES ('test');"))
            conn.execute(text("DROP TABLE connection_test;"))
            print("✅ Permissions: All necessary permissions are in place.")
            print("\n🚀 System is ready to proceed to the table creation stage!")

    except SQLAlchemyError as e:
        print("\n❌ System integrity check failed")
        print(f"⚠️ Technical reason: {e}")

if __name__ == "__main__":
    run_health_check()

In [ ]:
from sqlalchemy import (
    Column, Integer, String, Float, Date, Boolean,
    ForeignKey, Index, CheckConstraint
)
from sqlalchemy.orm import declarative_base, relationship

Base = declarative_base()

class Client(Base):
    __tablename__ = 'clients'
    
    client_id = Column(Integer, primary_key=True)
    score_credit_client = Column(Integer, CheckConstraint('score_credit_client >= 0'))
    segment_client = Column(String(100), nullable=False)


class Produit(Base):
    __tablename__ = 'produits'
    
    produit = Column(String(255), primary_key=True)
    categorie = Column(String(100), nullable=False)
    categorie_risque = Column(String(100))


class Agence(Base):
    __tablename__ = 'agences'
    
    agence = Column(String(255), primary_key=True)


class Temps(Base):
    __tablename__ = 'temps'
    
    date_transaction = Column(Date, primary_key=True)
    annee = Column(Integer, nullable=False)
    mois = Column(Integer, CheckConstraint('mois BETWEEN 1 AND 12'))
    trimestre = Column(Integer, CheckConstraint('trimestre BETWEEN 1 AND 4'))
    jour_semaine = Column(String(20))


class Transaction(Base):
    __tablename__ = 'transactions'
    
    transaction_id = Column(Integer, primary_key=True)

    client_id = Column(Integer, ForeignKey('clients.client_id'), nullable=False)
    produit = Column(String(255), ForeignKey('produits.produit'), nullable=False)
    agence = Column(String(255), ForeignKey('agences.agence'), nullable=False)
    date_transaction = Column(Date, ForeignKey('temps.date_transaction'), nullable=False)

    montant = Column(Float, nullable=False, CheckConstraint('montant > 0'))
    devise = Column(String(10), nullable=False)
    taux_change_eur = Column(Float, CheckConstraint('taux_change_eur > 0'))
    montant_eur = Column(Float, CheckConstraint('montant_eur >= 0'))

    type_operation = Column(String(100), nullable=False)
    statut = Column(String(50), nullable=False)

    is_anomaly = Column(Boolean, default=False)
    montant_eur_verifie = Column(Float)

    # العلاقات
    client = relationship("Client", backref="transactions")
    produit_rel = relationship("Produit", backref="transactions")
    agence_rel = relationship("Agence", backref="transactions")
    temps = relationship("Temps", backref="transactions")


# ✅ إنشاء الفهارس (Index)
Index('idx_client_id', Transaction.client_id)
Index('idx_date_transaction', Transaction.date_transaction)
Index('idx_agence', Transaction.agence)

In [ ]:
@event.listens_for(engine, "connect")
def set_webapp_encoding(dbapi_connection, connection_record):
    cursor = dbapi_connection.cursor()
    cursor.execute("SET client_encoding TO 'UTF8'")
    cursor.close()

try:
    Base.metadata.create_all(engine)
    print("🚀 All tables created successfully; all constraints and keys enforced.")
except Exception as e:
    print(f"❌ An error occurred while creating tables: {e}")

In [ ]:
import pandas as pd
import numpy as np

try:
    df = pd.read_csv('financecore_clean.csv', encoding='utf-8')
except Exception:
    df = pd.read_csv('financecore_clean.csv', encoding='latin1')

df['date_transaction'] = pd.to_datetime(df['date_transaction'])

In [ ]:
from sqlalchemy.orm import sessionmaker

Session = sessionmaker(bind=engine)
session = Session()

try:
    print("⏳ Starting Loading process ...")

    # --- Clients ---
    client_data = df[['client_id', 'score_credit_client', 'segment_client']].drop_duplicates('client_id')
    for _, r in client_data.iterrows():
        session.merge(Client(
            client_id           = int(r['client_id']),
            score_credit_client = int(r['score_credit_client']) if pd.notna(r['score_credit_client']) else None,
            segment_client      = r['segment_client']
        ))
    print("✅ Clients loaded successfully.")

    # --- Produits ---
    produit_data = df[['produit', 'categorie', 'categorie_risque']].drop_duplicates('produit')
    for _, r in produit_data.iterrows():
        session.merge(Produit(
            produit          = r['produit'],
            categorie        = r['categorie'],
            categorie_risque = r['categorie_risque']
        ))
    print("✅ Produits loaded successfully.")

    # --- Agences ---
    agence_data = df[['agence']].drop_duplicates('agence')
    for _, r in agence_data.iterrows():
        session.merge(Agence(agence=r['agence']))
    print("✅ Agences loaded successfully.")

    # --- Temps ---
    temps_data = df[['date_transaction']].drop_duplicates('date_transaction').copy()
    temps_data['annee']        = temps_data['date_transaction'].dt.year
    temps_data['mois']         = temps_data['date_transaction'].dt.month
    temps_data['trimestre']    = temps_data['date_transaction'].dt.quarter
    temps_data['jour_semaine'] = temps_data['date_transaction'].dt.day_name()
    for _, r in temps_data.iterrows():
        session.merge(Temps(
            date_transaction = r['date_transaction'].date(),
            annee            = int(r['annee']),
            mois             = int(r['mois']),
            trimestre        = int(r['trimestre']),
            jour_semaine     = r['jour_semaine']
        ))
    print("✅ Temps loaded successfully.")

    
    transaction_records = [
        {
            'transaction_id':      int(r['transaction_id']),
            'client_id':           int(r['client_id']),
            'produit':             r['produit'],
            'agence':              r['agence'],
            'date_transaction':    r['date_transaction'].date(),
            'montant':             float(r['montant']),
            'devise':              r['devise'],
            'taux_change_eur':     float(r['taux_change_eur'])     if pd.notna(r['taux_change_eur'])     else None,
            'montant_eur':         float(r['montant_eur'])         if pd.notna(r['montant_eur'])         else None,
            'type_operation':      r['type_operation'],
            'statut':              r['statut'],
            'is_anomaly':          bool(r['is_anomaly']),
            'montant_eur_verifie': float(r['montant_eur_verifie']) if pd.notna(r['montant_eur_verifie']) else None,
        }
        for _, r in df.iterrows()
    ]
    session.bulk_insert_mappings(Transaction, transaction_records)
    print("✅ Transactions loaded successfully.")

    session.commit()
    print("✨ All data has been loaded successfully!")

except Exception as e:
    session.rollback()
    print(f"❌ Failed to load data: {e}")
finally:
    session.close()

Vue 1 complète des transactions

In [ ]:
CREATE VIEW vue_transactions_detail AS
SELECT 
    t.transaction_id,
    c.client_id,
    c.segment_client,
    c.score_credit_client,
    
    p.produit,
    p.categorie,
    p.categorie_risque,
    
    a.agence,
    
    t.date_transaction,
    tp.annee,
    tp.mois,
    tp.trimestre,
    tp.jour_semaine,
    
    t.montant,
    t.devise,
    t.montant_eur,
    t.type_operation,
    t.statut,
    t.is_anomaly

FROM transactions t
JOIN clients c ON t.client_id = c.client_id
JOIN produits p ON t.produit = p.produit
JOIN agences a ON t.agence = a.agence
JOIN temps tp ON t.date_transaction = tp.date_transaction;

vue 2 Analyse par client

In [ ]:
CREATE VIEW vue_transactions_client AS
SELECT 
    c.client_id,
    c.segment_client,
    COUNT(t.transaction_id) AS nb_transactions,
    SUM(t.montant_eur) AS total_eur,
    AVG(t.montant_eur) AS moyenne_eur

FROM clients c
JOIN transactions t ON c.client_id = t.client_id
GROUP BY c.client_id, c.segment_client;

vue 3 Détection anomalies

In [ ]:
CREATE VIEW vue_anomalies AS
SELECT 
    t.transaction_id,
    t.client_id,
    t.montant_eur,
    t.montant_eur_verifie,
    t.statut

FROM transactions t
WHERE t.is_anomaly = TRUE
   OR t.montant_eur <> t.montant_eur_verifie;